# Semana 10 · Clase 1 — Línea base, modelo y evaluación honesta

**Electiva de Concentración I (Ciencia de Datos) · 2026-II · Unidad 2**

La semana pasada usted preparó las 400 tiendas de sillas para auto infantil (`Carseats`): partió la tabla en **train** y **test** con `SEMILLA = 1184`, rellenó vacíos con la mediana de train y armó un pipeline de preprocesamiento. Hoy da los dos pasos que siguen en el proceso de ciencia de datos: **modelar** y **evaluar con honestidad**.

La regla del día: **un modelo solo sirve si se desempeña mejor que la línea base ingenua, cuando se utilizan datos que el modelo no vio.**

**Herramienta nueva de hoy:** ninguna. Todo sale de `scikit-learn`, que usó la semana pasada. `joblib`, para guardar el modelo, viene instalado con `scikit-learn`.

**Qué entrega (Taller 14, sobre 20 puntos):** este mismo cuaderno, con todos los `# TODO` y la Parte B completos, ejecutado con `Kernel → Restart & Run All` y guardado con las salidas visibles, con el nombre exacto:

`Taller 14 - Electiva de Concentración I - Ciencia de Datos 2026-II - modelo contra línea base.ipynb`

**Fecha límite: hoy, martes 10 de noviembre de 2026**, en Brightspace. 

**Archivos que necesita en esta misma carpeta:** `Carseats_faltantes.csv` y `Auto.csv`.

**Archivos que este cuaderno escribe, y que el jueves vuelve a leer (no los borre):**

| Archivo | Lo escribe | Lo usa |
|---|---|---|
| `resultado_carseats.csv` | Parte 5 | jueves 12-nov, tablero de ejemplo |
| `modelo_carseats.joblib` | Parte 5 | práctica de refuerzo de hoy |
| `datos_proyecto.csv` | Parte B | jueves 12-nov y tablero del proyecto final |
| `resultado_proyecto.csv` | Parte B | jueves 12-nov y tablero del proyecto final |
| `modelo_proyecto.joblib` | Parte B | martes 17-nov, vista de predicción |

---
## Parte 0 — Retomar la partición de la semana pasada

Misma tabla, misma semilla, mismas filas en test. Si los números de esta parte no coinciden con los de la semana 9, nada de lo que sigue será comparable.

In [1]:
import pandas as pd  # tablas de datos
sucio = pd.read_csv('Carseats_faltantes.csv')  # 400 tiendas, con celdas vacias
X = sucio.drop(columns='Sales')  # columnas de entrada
y = sucio['Sales']  # lo que se predice: ventas, en miles de unidades
print(sucio.shape)

(400, 11)


In [2]:
from sklearn.model_selection import train_test_split  # particion train/test
SEMILLA = 1184  # la misma semilla de la semana 9
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEMILLA)
print(X_train.shape, X_test.shape)

(300, 10) (100, 10)


In [3]:
# Autochequeo: la particion debe ser la misma de la semana 9.
assert X_train.shape == (300, 10) and X_test.shape == (100, 10), 'revise test_size=0.25'
assert list(X_test.index[:3]) == [116, 259, 313], 'la semilla no es 1184'

---
## Parte 1 — La línea base ingenua

Antes de entrenar cualquier modelo, mida la predicción más ingenua posible: **decir siempre el promedio de train**, para todas las tiendas. No usa ninguna columna. Si un modelo no le gana a eso, no sirve para decidir nada.

Para medir el error se usa el **MAE** (*mean absolute error*, error absoluto medio): el promedio de cuánto se equivoca la predicción, sin importar el signo. Está en las mismas unidades que `Sales` (miles de unidades vendidas).

Prediga antes de correr: ¿Cuánto se equivocará, en promedio, la línea base en test? ¿Más o menos de 1 (mil unidades)?

Mi predicción: _En promedio la línea base en test se equivocara por lo menos 1200 unidades_

In [4]:
media_train = y_train.mean()  # el promedio sale solo de train
pred_base = [media_train] * len(y_test)  # la misma prediccion para cada tienda de test
print('promedio de train:', round(media_train, 2))

promedio de train: 7.48


In [5]:
from sklearn.metrics import mean_absolute_error  # error absoluto medio
mae_base = mean_absolute_error(y_test, pred_base)  # se mide en test
print('MAE de la linea base:', round(mae_base, 3))

MAE de la linea base: 2.274


In [6]:
# Autochequeo: el valor depende de la particion con SEMILLA = 1184.
assert round(media_train, 2) == 7.48, 'el promedio debe salir de y_train'
assert round(mae_base, 3) == 2.274, f'se esperaba 2.274, hay {mae_base:.3f}'

`scikit-learn` trae la misma línea base ya hecha: `DummyRegressor` (*regresor ingenuo*). Hace exactamente lo que usted hizo a mano. La usará en la Parte B.

In [7]:
from sklearn.dummy import DummyRegressor  # linea base: predice siempre el promedio
base = DummyRegressor(strategy='mean').fit(X_train, y_train)  # aprende el promedio de train
print('MAE con DummyRegressor:', round(mean_absolute_error(y_test, base.predict(X_test)), 3))

MAE con DummyRegressor: 2.274


**`# TODO 1`** — Con sus números: ¿Qué significa, para el gerente de una tienda, un MAE de 2.27? Use la unidad de `Sales`.

_Para un gerente una MAE significa que el error esta alojado en 2.27 unidades en ventas_

### 🅢 Punto de encuentro 1 — línea base y MAE *(pausa guiada)*

- **Línea base:** la predicción más simple que no usa ninguna columna. Es la vara contra la que se mide cualquier modelo.
- **MAE:** cuánto se equivoca la predicción, en promedio, en las unidades del problema.

*"Sin línea base, un MAE de 1 no dice nada: no sabemos si es bueno o malo. Con línea base, sí."*

---
## Parte 2 — Un modelo que tiene que ganarle

Ahora sí, un modelo: **regresión lineal**. Combina las columnas con un peso para cada una. Se entrena sobre el mismo preprocesamiento de la semana 9. Para no repetir código, el preprocesamiento queda dentro de una función que crea uno nuevo cada vez que la llama.

Prediga antes de correr: ¿El MAE del modelo en test será menor que 2.27? ¿Cuánto, más o menos?

Mi predicción: _Predigo que el MAE será menor que 2.27, menor unos 1.30, es decir un valor de 0.97_

In [8]:
from sklearn.pipeline import Pipeline  # cadena de pasos
from sklearn.compose import ColumnTransformer  # un tratamiento por tipo de columna
from sklearn.impute import SimpleImputer  # rellena vacios
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # escala y categorias
numericas = ['CompPrice', 'Income', 'Advertising', 'Population', 'Price', 'Age', 'Education']
categoricas = ['ShelveLoc', 'Urban', 'US']

In [9]:
def crear_preproc(numericas, categoricas):
    # Devuelve un preprocesamiento nuevo, igual al de la semana 9.
    tratar_num = Pipeline([('relleno', SimpleImputer(strategy='median')),
                           ('escala', StandardScaler())])
    tratar_cat = Pipeline([('relleno', SimpleImputer(strategy='most_frequent')),
                           ('onehot', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('num', tratar_num, numericas),
                              ('cat', tratar_cat, categoricas)])

In [10]:
from sklearn.linear_model import LinearRegression  # regresion lineal
modelo = Pipeline([('preproc', crear_preproc(numericas, categoricas)),
                   ('regresion', LinearRegression())])
modelo.fit(X_train, y_train)  # aprende solo de train
mae_modelo = mean_absolute_error(y_test, modelo.predict(X_test))  # se mide en test
print('MAE del modelo:', round(mae_modelo, 3), '| linea base:', round(mae_base, 3))

MAE del modelo: 0.978 | linea base: 2.274


In [11]:
# Autochequeo: el modelo le gana a la linea base, en test.
assert round(mae_modelo, 3) == 0.978, f'se esperaba 0.978, hay {mae_modelo:.3f}'
assert mae_modelo < mae_base, 'el modelo deberia ganarle a la linea base'

**`# TODO 2`** — Compare los dos MAE con sus números. ¿Vale la pena el modelo frente a la línea base? Responda en una o dos frases.

_Sí, el MAE del modelo tiene un valor mucho más inferior que el MAE de la línea base por lo tanto se puede decir que es más preciso_

---
## Parte 3 — La métrica sospechosamente perfecta

Un **árbol de decisión** sin límite de profundidad puede memorizar cada tienda de train. Vamos a medirlo dos veces: sobre train (lo que ya vio) y sobre test (lo que no vio).

Prediga antes de correr: ¿Cuánto dará el MAE del árbol **en train**?

Mi predicción: _El MAE del arbol será de 3.56_

In [12]:
from sklearn.tree import DecisionTreeRegressor  # arbol de decision
arbol = Pipeline([('preproc', crear_preproc(numericas, categoricas)),
                  ('arbol', DecisionTreeRegressor(random_state=SEMILLA))])
arbol.fit(X_train, y_train)  # aprende solo de train
mae_arbol_train = mean_absolute_error(y_train, arbol.predict(X_train))  # medido en lo que ya vio
mae_arbol_test = mean_absolute_error(y_test, arbol.predict(X_test))  # medido en lo que no vio
print('arbol en train:', round(mae_arbol_train, 3), '| arbol en test:', round(mae_arbol_test, 3))

arbol en train: 0.0 | arbol en test: 1.681


In [13]:
# Autochequeo: perfecto en train, peor que la regresion en test.
assert mae_arbol_train == 0, 'el arbol sin limite memoriza train'
assert round(mae_arbol_test, 3) == 1.681 and mae_arbol_test > mae_modelo

**`# TODO 3`** — Si solo hubiera medido en train, ¿Qué modelo habría elegido? ¿Por qué esa elección habría sido un error? Use sus números.

_hubiera escogio el MAE del arbol porque su valor ha sido 0.0, aunque hubiera sido un error por sobreajuste debido a que luego el arbol en test empeoró al evaluar esos nuevos datos mostrando una regresión de 1.681_

### 🅢 Punto de encuentro 2 — evaluar con honestidad *(pausa guiada)*

- **Train** sirve para aprender; **test**, para evaluar. Una métrica medida en train no dice cómo le irá al modelo con datos nuevos.
- Un resultado perfecto es una **alarma**, no una buena noticia.

*"Si su modelo da un error de cero, no lo celebre: revise dónde lo midió."*

---
## Parte 4 — Cuando la respuesta es una categoría

Algunos proyectos no predicen un número sino una **categoría** (sí/no, alto/bajo). Con `Carseats` se puede preguntar: ¿La tienda vende más de 8 mil unidades? Eso es **clasificación** (semana 6).

Cambian dos cosas: la línea base es **decir siempre la clase más frecuente** de train y la métrica es la **exactitud**: la proporción de tiendas de test que el modelo acierta.

Prediga antes de correr: ¿Qué exactitud tendrá la línea base, que nunca mira ninguna columna?

Mi predicción: _Yo creo que el valor estará entre 0.6 - 1.2_

In [14]:
alta_train = y_train > 8  # True si la tienda vende mas de 8 mil unidades
alta_test = y_test > 8
print('fraccion de tiendas con ventas altas en train:', round(alta_train.mean(), 3))

fraccion de tiendas con ventas altas en train: 0.403


In [15]:
from sklearn.dummy import DummyClassifier  # linea base: siempre la clase mas frecuente
from sklearn.metrics import accuracy_score  # exactitud: proporcion de aciertos
base_clas = DummyClassifier(strategy='most_frequent').fit(X_train, alta_train)
exactitud_base = accuracy_score(alta_test, base_clas.predict(X_test))  # se mide en test
print('exactitud de la linea base:', exactitud_base)

exactitud de la linea base: 0.57


In [16]:
from sklearn.linear_model import LogisticRegression  # regresion logistica: clasifica
logistica = Pipeline([('preproc', crear_preproc(numericas, categoricas)),
                      ('logistica', LogisticRegression(max_iter=1000))])
logistica.fit(X_train, alta_train)  # aprende solo de train
exactitud_modelo = accuracy_score(alta_test, logistica.predict(X_test))  # se mide en test
print('exactitud del modelo:', exactitud_modelo, '| linea base:', exactitud_base)

exactitud del modelo: 0.83 | linea base: 0.57


In [17]:
# Autochequeo: la linea base ya acierta 57 de 100; el modelo, 83.
assert exactitud_base == 0.57, f'se esperaba 0.57, hay {exactitud_base}'
assert exactitud_modelo == 0.83, f'se esperaba 0.83, hay {exactitud_modelo}'

**`# TODO 4`** — Alguien le dice: "mi modelo acierta el 57 %". Con lo que acaba de ver, ¿Eso es bueno? ¿Qué número le pediría para saberlo?

_No es bueno, solo predice la clase más frecuente, si uno desea saberlo se necesita la excatitud de la línea base para verificar si el modelo aporta un valor predictivo._

---
## Parte 5 — Guardar el modelo y el resultado

Un modelo que vive solo en la memoria del cuaderno se pierde al cerrar Jupyter. `joblib.dump` lo guarda en un archivo y `joblib.load` lo trae de vuelta, ya entrenado. El resultado de la evaluación se guarda en un CSV de una fila: Después, el tablero interactivo lo lee de ahí en lugar de volver a calcularlo.

In [18]:
import joblib  # guarda y carga modelos entrenados
joblib.dump(modelo, 'modelo_carseats.joblib')  # el pipeline completo: preproc + regresion
print('modelo guardado')

modelo guardado


In [19]:
resultado_carseats = pd.DataFrame([{
    'dataset': 'Carseats_faltantes.csv', 'objetivo': 'Sales',
    'tipo_tarea': 'regresion', 'metrica': 'MAE',
    'valor_linea_base': round(mae_base, 3), 'valor_modelo': round(mae_modelo, 3),
    'filas_train': len(X_train), 'filas_test': len(X_test),
}])  # una sola fila con lo que se midio
resultado_carseats.to_csv('resultado_carseats.csv', index=False)  # lo leerá posteriormente lo lee
resultado_carseats

,dataset,objetivo,tipo_tarea,metrica,valor_linea_base,valor_modelo,filas_train,filas_test
0,Carseats_faltantes.csv,Sales,regresion,MAE,2.274,0.978,300,100


In [20]:
# Autochequeo: el archivo quedo en disco y se lee igual.
leido = pd.read_csv('resultado_carseats.csv')
assert leido.loc[0, 'valor_modelo'] == 0.978 and leido.loc[0, 'valor_linea_base'] == 2.274

---
## Parte B — Ahora usted: el mismo procedimiento con su proyecto

Hasta aquí trabajó con `Carseats`. Ahora repite el procedimiento de las Partes 1, 2 y 5 con **el dataset de su proyecto**, el que perfiló (semana 7), graficó (semana 8) y limpió (semana 9). Lo que produzca aquí es la base del tablero del jueves y del proyecto final.

**Si ya tiene su dataset:** cópielo a esta carpeta y escriba su nombre en `RUTA_DATASET_PROPIO`.

**Si todavía no lo tiene listo:** use `Auto.csv` como interino (predecir `mpg`, rendimiento en millas por galón) y dígalo en el `# TODO 6`. No lo esconda: antes del jueves 26 de noviembre debe repetir esta parte con su dataset real.

**Reglas:**
- `OBJETIVO` es la columna que su pregunta quiere predecir.
- `TIPO_TAREA` es `'regresion'` si el objetivo es un número, o `'clasificacion'` si es una categoría.
- `NUMERICAS_B` y `CATEGORICAS_B` son listas con las columnas de entrada. Nunca ponga el objetivo en ellas. Si no tiene columnas categóricas, deje la lista vacía: `[]`.
- Use la misma `SEMILLA` y `test_size=0.25`.

**Qué se califica:** las celdas de la Parte B ejecutadas, los tres archivos escritos y los `# TODO 6` y `# TODO 7` respondidos con sus números.

**`# TODO 5` — Ahora usted (defina su problema):** complete los cinco valores de la celda de abajo.

In [21]:
RUTA_DATASET_PROPIO = 'bedtime_screentime_sleep_debt.csv'  # reemplace por el nombre de su archivo si ya lo tiene
OBJETIVO = 'total_sleep_hours'  # la columna que quiere predecir, entre comillas
TIPO_TAREA = 'regresion'  # 'regresion' o 'clasificacion'
NUMERICAS_B = ['bedtime_phone_minutes']  # lista de columnas numericas de entrada
CATEGORICAS_B = []  # lista de columnas categoricas de entrada (puede ser [])

In [22]:
columnas = [OBJETIVO] + NUMERICAS_B + CATEGORICAS_B  # solo lo que usa el modelo
datos = pd.read_csv(RUTA_DATASET_PROPIO)[columnas]
datos = datos.dropna(subset=[OBJETIVO])  # sin objetivo no hay nada que aprender ni evaluar
print(datos.shape)
datos.head(3)

(8500, 2)


,total_sleep_hours,bedtime_phone_minutes
0,3.20,179
1,4.06,163
2,3.20,100


In [23]:
# Autochequeo: el problema esta bien definido.
assert TIPO_TAREA in ['regresion', 'clasificacion'], "escriba 'regresion' o 'clasificacion'"
assert OBJETIVO not in NUMERICAS_B + CATEGORICAS_B, 'el objetivo no puede ser una entrada'
for col in NUMERICAS_B:
    assert pd.api.types.is_numeric_dtype(datos[col]), f'{col} no es numerica: conviertala (semana 3) o pasela a CATEGORICAS_B'
assert len(datos) >= 50, 'con menos de 50 filas no hay suficiente para train y test'

In [24]:
X_b = datos.drop(columns=OBJETIVO)  # entradas
y_b = datos[OBJETIVO]  # objetivo
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, test_size=0.25, random_state=SEMILLA)
print(X_train_b.shape, X_test_b.shape)

(6375, 1) (2125, 1)


La celda de abajo elige la línea base, el modelo y la métrica según el tipo de tarea, igual que en las Partes 1, 2 y 4. Léala antes de correrla: cada línea es algo que ya usó hoy.

In [25]:
if TIPO_TAREA == 'regresion':
    base_b = DummyRegressor(strategy='mean')  # siempre el promedio de train
    final_b = LinearRegression()
    metrica = 'MAE'
else:
    base_b = DummyClassifier(strategy='most_frequent')  # siempre la clase mas frecuente
    final_b = LogisticRegression(max_iter=1000)
    metrica = 'exactitud'
modelo_b = Pipeline([('preproc', crear_preproc(NUMERICAS_B, CATEGORICAS_B)), ('modelo', final_b)])
print('linea base:', base_b, '| modelo:', final_b, '| metrica:', metrica)

linea base: DummyRegressor() | modelo: LinearRegression() | metrica: MAE


In [26]:
base_b.fit(X_train_b, y_train_b)  # las dos cosas aprenden solo de train
modelo_b.fit(X_train_b, y_train_b)
if metrica == 'MAE':
    valor_base = mean_absolute_error(y_test_b, base_b.predict(X_test_b))  # las dos se miden en test
    valor_modelo = mean_absolute_error(y_test_b, modelo_b.predict(X_test_b))
else:
    valor_base = accuracy_score(y_test_b, base_b.predict(X_test_b))
    valor_modelo = accuracy_score(y_test_b, modelo_b.predict(X_test_b))
print(metrica, '| linea base:', round(valor_base, 3), '| modelo:', round(valor_modelo, 3))

MAE | linea base: 1.211 | modelo: 1.002


In [27]:
# Chequeo: no exige que el modelo gane; exige que usted lo sepa.
if metrica == 'MAE':
    gana = valor_modelo < valor_base  # menos error es mejor
else:
    gana = valor_modelo > valor_base  # mas aciertos es mejor
print('El modelo le gana a la linea base:', gana)

El modelo le gana a la linea base: True


**`# TODO 6`** — Con sus números: ¿Su modelo le gana a la línea base? y ¿Por cuánto? Compárelo con `Carseats` (0.98 contra 2.27). Diga qué dataset usó; si fue `Auto.csv` como interino, dígalo. Si su modelo **no** gana, no lo esconda: diga qué haría para mejorarlo. Documentar un fallo también puntúa.

_utilizando el dataset (bedtime_screentime_sleep_debt.csv), el modelo obtuvo un MAE de 1002 frente a 1211 de la linea base, lo que practicamente representa una reducción del error del 0.209._

Ahora guarde los tres archivos que el jueves necesita. Son la conexión entre esta clase y la siguiente: el tablero interactivo los lee tal cual.

In [28]:
datos.to_csv('datos_proyecto.csv', index=False)  # los datos que usa el modelo
joblib.dump(modelo_b, 'modelo_proyecto.joblib')  # el modelo entrenado
resultado_proyecto = pd.DataFrame([{
    'dataset': RUTA_DATASET_PROPIO, 'objetivo': OBJETIVO,
    'tipo_tarea': TIPO_TAREA, 'metrica': metrica,
    'valor_linea_base': round(valor_base, 3), 'valor_modelo': round(valor_modelo, 3),
    'filas_train': len(X_train_b), 'filas_test': len(X_test_b),
}])  # mismas columnas que resultado_carseats.csv
resultado_proyecto.to_csv('resultado_proyecto.csv', index=False)
resultado_proyecto

,dataset,objetivo,tipo_tarea,metrica,valor_linea_base,valor_modelo,filas_train,filas_test
0,bedtime_screentime_sleep_debt.csv,total_sleep_hours,regresion,MAE,1.211,1.002,6375,2125


In [29]:
# Autochequeo: los tres archivos existen y se leen igual.
from pathlib import Path  # para preguntar si un archivo existe
for nombre in ['datos_proyecto.csv', 'resultado_proyecto.csv', 'modelo_proyecto.joblib']:
    assert Path(nombre).exists(), f'falta {nombre}'
assert pd.read_csv('resultado_proyecto.csv').loc[0, 'valor_modelo'] == round(valor_modelo, 3)
assert pd.read_csv('datos_proyecto.csv').shape == datos.shape
print('Los tres archivos del contrato estan listos para el jueves.')

Los tres archivos del contrato estan listos para el jueves.


**`# TODO 7`** — El jueves este resultado va a un tablero. ¿Qué decisión ayuda a tomar, y quién la tomaría? Una o dos frases, sobre su proyecto.

_El resultado ayuda a estimar a los profesionales del bienestar sobre cúantas horas de descanso tendrán las personas según los habitos nocturnos de pantalla, con esto las personas que se encargan de la salud pueden tomar decisiones para ajustar las horarios de uso de dispositivos antes de dormir para mejorar su calidad de vida_

---
## Práctica de refuerzo

Dos ejercicios cortos, cada uno con su autochequeo. Son parte del mismo entregable.

**Refuerzo 1 — el modelo vuelve del disco.** Cargue `modelo_carseats.joblib` con `joblib.load` y guárdelo en `modelo_cargado`. Luego prediga las ventas de la **primera tienda** de `X_test` (use `X_test.head(1)`) y guarde ese número en `prediccion_cargada`.

In [ ]:
modelo_cargado = ____  # joblib.load con el nombre del archivo
prediccion_cargada = ____  # .predict(X_test.head(1)) y tome el elemento [0]
print(round(prediccion_cargada, 3))

In [ ]:
# Autochequeo: el modelo cargado predice lo mismo que el que esta en memoria.
assert round(prediccion_cargada, 3) == round(modelo.predict(X_test.head(1))[0], 3) == 5.123

**Refuerzo 2 — la mejora en porcentaje.** Calcule cuánto baja el error del modelo frente a la línea base en `Carseats`, en porcentaje: `(mae_base - mae_modelo) / mae_base * 100`, redondeado a un decimal con `round(..., 1)`. Guárdelo en `mejora`.

In [ ]:
mejora = ____  # porcentaje, con un decimal
print(mejora, '%')

In [ ]:
# Autochequeo: el modelo reduce el error de la linea base en 57 por ciento.
assert mejora == 57.0, f'se esperaba 57.0, hay {mejora}'

---
## Cierre

**Antes de entregar:** guarde con `Ctrl+S`, corra `Kernel → Restart & Run All` y verifique que **ninguna celda termine en error**. Este cuaderno no tiene errores intencionales: si una celda falla, la causa es un `____` sin completar o una columna mal escrita. Si se queda atascado en una celda, siga con las de abajo (`Cell → Run All Below`) y vuelva después: quedarse parado cuesta más que el error.

**No borre** los archivos que escribió (`resultado_carseats.csv`, `datos_proyecto.csv`, `resultado_proyecto.csv`, los dos `.joblib`): el jueves los necesita. Si trabaja en un computador del salón, quedan en su cuenta institucional.

**Cómo se califica:**
- Partición con la semilla de la semana 9 y línea base con su MAE en test, con el `# TODO 1` (Partes 0 y 1).
- Regresión lineal que supera a la línea base en test, con el `# TODO 2` (Parte 2).
- Métrica perfecta en train detectada (`# TODO 3`) y línea base de clasificación (`# TODO 4`) (Partes 3 y 4).
- Parte B: su problema definido, línea base y modelo medidos en test con la misma partición, autochequeos superados.
- Parte B: los tres archivos del contrato escritos y los `# TODO 6` y `# TODO 7` respondidos con sus números (declarando si usó `Auto.csv`).
- Práctica de refuerzo y cuaderno ejecutado completo, sin errores y con salidas guardadas.

No se acepta entregar en blanco, afirmar que algo funciona sin evidencia, ni decir que "no le sirvió" sin mostrar el texto del error. Se califica el razonamiento con los números de su propia ejecución, no una respuesta genérica.

In [42]:
# Resumen de su ejecucion: revise que cada linea tenga un numero.
print('Carseats  | linea base MAE:', round(mae_base, 3), '| modelo MAE:', round(mae_modelo, 3))
print('Arbol     | train:', round(mae_arbol_train, 3), '| test:', round(mae_arbol_test, 3))
print('Categoria | linea base:', exactitud_base, '| modelo:', exactitud_modelo)
print('Proyecto  |', RUTA_DATASET_PROPIO, '|', OBJETIVO, '|', metrica,
      '| linea base:', round(valor_base, 3), '| modelo:', round(valor_modelo, 3))

Carseats  | linea base MAE: 2.274 | modelo MAE: 0.978
Arbol     | train: 0.0 | test: 1.681
Categoria | linea base: 0.57 | modelo: 0.83
Proyecto  | bedtime_screentime_sleep_debt.csv | total_sleep_hours | MAE | linea base: 1.211 | modelo: 1.002
